In [1]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
 
print(train.head())
print(train.info())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

3. Data Preprocessing

In [15]:

#Handle Missing Values
train["Age"] = train["Age"].fillna(train["Age"].median())
train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])
train["Fare"] = train["Fare"].fillna(train["Fare"].median())


In [16]:
#Encode Categorical Columns
le = LabelEncoder()
train["Sex"] = le.fit_transform(train["Sex"])
train["Embarked"] = le.fit_transform(train["Embarked"])
train["Title"] = le.fit_transform(train["Title"])

In [17]:
#Feature Engineering
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
 
train["Title"] = train["Name"].str.extract(r' ([A-Za-z]+)\.', expand=False)
train["Title"] = train["Title"].fillna("Unknown")
 
le_title = LabelEncoder()
train["Title"] = le_title.fit_transform(train["Title"])

In [18]:
#Select Features
X = train[
[
"Pclass",
"Sex",
"Age",
"Fare",
"Embarked",
"FamilySize",
"Title"
]
]
 
y = train["Survived"]

In [20]:

#Normalize Features
print(X.isnull().sum())
print(X.isna().sum())

Pclass        0
Sex           0
Age           0
Fare          0
Embarked      0
FamilySize    0
Title         0
dtype: int64
Pclass        0
Sex           0
Age           0
Fare          0
Embarked      0
FamilySize    0
Title         0
dtype: int64


In [22]:
#normalise features
scaler = StandardScaler()
 
X = scaler.fit_transform(X)

In [23]:
#Cell 9: Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [24]:
#Convert to PyTorch Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
 
y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [27]:
#Cell 11: Check for NaN
print(torch.isnan(X_train).sum())
print(torch.isnan(y_train).sum())
#Both must print
#tensor(0)
#tensor(0)

tensor(0)
tensor(0)


In [30]:
#Task 1: Logistic Regression
model = nn.Sequential(
    nn.Linear(7,1),
    nn.Sigmoid()
)
 
criterion = nn.BCELoss()
 
optimizer = optim.SGD(model.parameters(), lr=0.01)
 
for epoch in range(100):
 
    output = model(X_train)
 
    loss = criterion(output, y_train)
 
    optimizer.zero_grad()
 
    loss.backward()
 
    optimizer.step()
 
print("Training Completed")

Training Completed


In [31]:
#Evaluate
with torch.no_grad():
 
    pred = model(X_test)
 
    pred = (pred >= 0.5).float()
 
print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1 Score :", f1_score(y_test, pred))

Accuracy : 0.7150837988826816
Precision: 0.6493506493506493
Recall   : 0.6756756756756757
F1 Score : 0.6622516556291391
